# 🧹 Phase 2 — Data Cleaning
### SoundMetrics B2C SaaS Customer Segmentation Project
**Author:** Radheshyam  
**Goal:** Take the raw, messy generated data and produce clean, analysis-ready CSVs.

---

## Why Data Cleaning Matters

In real companies, raw data is almost always messy. As a data analyst, your job is to:
1. **Find** all the problems in the data
2. **Decide** how to fix each problem (don't just delete everything!)
3. **Document** every decision you make (this is what separates juniors from seniors)
4. **Save** the clean data for the next phase

---

## What We Will Clean Today

| Table | Problems to Fix |
|---|---|
| users | Nulls, duplicate rows, impossible ages, mixed-case names |
| subscriptions | Nulls, wrong prices, end_date before start_date |
| listening_events | Nulls, negative/outlier durations |
| payments | Nulls, duplicate rows, suspicious amounts |
| support_tickets | Nulls, resolved_date before created_date, scores on unresolved tickets |

---

## 0️⃣ Setup — Import Libraries & Load Data

First we import the tools we need, then load all 5 raw CSV files.

**`pandas`** — the main library for working with tabular data (like Excel but in Python)  
**`numpy`** — used for maths and handling missing values  
**`os`** — used to create folders and manage file paths  

In [3]:
import pandas as pd
import numpy as np
import os

# Where our raw files live
RAW_DIR     = 'data/raw'

# Where we will save the cleaned files
CLEAN_DIR   = 'data/cleaned'
os.makedirs(CLEAN_DIR, exist_ok=True)

# ── Load all 5 tables ─────────────────────────────────────
users   = pd.read_csv(f'{RAW_DIR}/users.csv')
subs    = pd.read_csv(f'{RAW_DIR}/subscriptions.csv')
events  = pd.read_csv(f'{RAW_DIR}/listening_events.csv')
pays    = pd.read_csv(f'{RAW_DIR}/payments.csv')
tickets = pd.read_csv(f'{RAW_DIR}/support_tickets.csv')

print('✅ All 5 tables loaded successfully!')
print(f'  users          : {len(users):>10,} rows')
print(f'  subscriptions  : {len(subs):>10,} rows')
print(f'  listening_events: {len(events):>9,} rows')
print(f'  payments       : {len(pays):>10,} rows')
print(f'  support_tickets: {len(tickets):>10,} rows')

✅ All 5 tables loaded successfully!
  users          :    101,500 rows
  subscriptions  :    100,000 rows
  listening_events: 1,186,834 rows
  payments       :     80,960 rows
  support_tickets:     20,000 rows


---
## 1️⃣ Data Audit — Find All Problems First

> 💡 **Rule #1 of Data Cleaning:** Never start fixing things before you fully understand what's broken.

We first run a full audit on every table. This is called **Exploratory Data Quality Check (EDQC)**.
This is exactly what senior analysts do on Day 1 with any new dataset.

In [4]:
def audit_table(df, name):
    """
    Prints a full quality report for a dataframe.
    Shows: shape, dtypes, null counts, and duplicate count.
    """
    print(f'\n{'='*55}')
    print(f'  TABLE: {name}')
    print(f'{'='*55}')
    print(f'  Shape     : {df.shape[0]:,} rows × {df.shape[1]} columns')
    print(f'  Duplicates: {df.duplicated().sum():,} duplicate rows')
    print(f'\n  Column-wise Null Report:')
    print(f'  {"Column":<28} {"Dtype":<12} {"Nulls":>8} {"Null%":>8}')
    print(f'  {"-"*60}')
    for col in df.columns:
        nulls = df[col].isnull().sum()
        pct   = nulls / len(df) * 100
        flag  = ' ⚠️' if pct > 5 else ''
        print(f'  {col:<28} {str(df[col].dtype):<12} {nulls:>8,} {pct:>7.1f}%{flag}')

# Run audit on all tables
for df, name in [(users,'users'),(subs,'subscriptions'),
                  (events,'listening_events'),(pays,'payments'),
                  (tickets,'support_tickets')]:
    audit_table(df, name)


  TABLE: users
  Shape     : 101,500 rows × 11 columns
  Duplicates: 1,500 duplicate rows

  Column-wise Null Report:
  Column                       Dtype           Nulls    Null%
  ------------------------------------------------------------
  user_id                      int64               0     0.0%
  full_name                    object              0     0.0%
  email                        object          3,030     3.0%
  age                          float64         5,164     5.1% ⚠️
  gender                       object          7,044     6.9% ⚠️
  country                      object              0     0.0%
  currency                     object              0     0.0%
  signup_date                  object              0     0.0%
  acquisition_channel          object              0     0.0%
  user_segment                 object              0     0.0%
  is_bot                       int64               0     0.0%

  TABLE: subscriptions
  Shape     : 100,000 rows × 10 columns
  Du

In [5]:
# ── Check sample rows to see what the data looks like ─────
print('--- USERS sample ---')
print(users.head(3).to_string())
print('\n--- SUBSCRIPTIONS sample ---')
print(subs.head(3).to_string())

--- USERS sample ---
   user_id         full_name                     email   age gender        country currency signup_date  acquisition_channel  user_segment  is_bot
0        1  Sebastian Garcia  sebastian3888@icloud.com  15.0   Male        Germany      EUR  2023-08-09  App Store - Android   Casual User       0
1        2   Babatunde Banda    babatunde5383@mail.com  31.0   Male       Ethiopia      ETB  2020-03-25  App Store - Android  Dormant User       0
2        3    Benjamin Allen     benjamin1707@zoho.com  22.0   Male  United States      USD  2022-10-10     Paid Ad - Google  Dormant User       0

--- SUBSCRIPTIONS sample ---
   sub_id  user_id plan_type previous_plan     status  start_date    end_date  monthly_price currency  auto_renew
0       1        1   Student       Premium     Paused  2023-08-24  2025-05-29           2.75      EUR           1
1       2        2      Free           NaN  Cancelled  2020-04-07  2022-01-21           0.00      ETB           0
2       3        3 

---
## 2️⃣ Cleaning: `users` Table

**Problems we know exist:**
- Duplicate rows (~1.5%)
- Null emails (~3%), null ages (~5%), null genders (~7%)
- Impossible ages: 0, -1, 150, 999
- Mixed-case names: 'JOHN SMITH' or 'john smith'
- Bot accounts (is_bot = 1)

**Our decisions:**
| Problem | Decision | Why |
|---|---|---|
| Duplicate rows | Remove | Same user shouldn't appear twice |
| Null email | Keep row, flag as 'unknown' | User is still valid |
| Null age | Keep row, fill with median | Age is useful but not critical |
| Null gender | Keep row, fill with 'Unknown' | Respect user privacy |
| Impossible ages | Replace with NaN then median | Can't fix what we don't know |
| Mixed case names | Standardise to Title Case | 'John Smith' is the standard |
| Bot accounts | Flag separately, remove from main | Bots skew all analysis |

In [6]:
# ── Step 2.1: Remove duplicate rows ──────────────────────
# .duplicated() returns True for rows that are exact copies
# keep='first' means we keep the first occurrence and drop the rest

before = len(users)
users  = users.drop_duplicates(keep='first')
after  = len(users)

print(f'Duplicates removed : {before - after:,}')
print(f'Rows remaining     : {after:,}')

Duplicates removed : 1,500
Rows remaining     : 100,000


In [7]:
# ── Step 2.2: Separate bot accounts ──────────────────────
# We save bots to their own file — useful for a 'Fraud Detection'
# section in your portfolio README

bots  = users[users['is_bot'] == 1].copy()
users = users[users['is_bot'] == 0].copy()

# Drop the is_bot column — no longer needed in main table
users = users.drop(columns=['is_bot'])

print(f'Bot accounts found and removed : {len(bots):,}')
print(f'Clean human users remaining    : {len(users):,}')

# Save bots separately
bots.to_csv(f'{CLEAN_DIR}/bot_accounts.csv', index=False)
print(f'Bot accounts saved to: {CLEAN_DIR}/bot_accounts.csv')

Bot accounts found and removed : 500
Clean human users remaining    : 99,500
Bot accounts saved to: data/cleaned/bot_accounts.csv


In [8]:
# ── Step 2.3: Fix impossible ages ────────────────────────
# First let's see what impossible values exist
print('Age value counts for suspicious values:')
print(users[users['age'].notna() & ~users['age'].between(13, 100)]['age'].value_counts())

# Replace impossible ages with NaN
# .between(13, 100) returns True if value is within 13-100
# ~ means NOT — so we're selecting ages outside 13-100
users.loc[~users['age'].between(13, 100, inclusive='both'), 'age'] = np.nan

# Now fill all NaN ages with the median age
# Median is better than mean when data has outliers
median_age = users['age'].median()
users['age'] = users['age'].fillna(median_age)
users['age'] = users['age'].astype(int)   # Convert to whole number

print(f'\nMedian age used for filling: {median_age:.0f}')
print(f'Age range after cleaning   : {users["age"].min()} - {users["age"].max()}')

Age value counts for suspicious values:
age
 150.0    55
-1.0      55
 999.0    50
 0.0      39
Name: count, dtype: int64

Median age used for filling: 28
Age range after cleaning   : 13 - 70


In [9]:
# ── Step 2.4: Fix null emails ─────────────────────────────
# We can't guess someone's email — so we fill with 'unknown'
# This preserves the row while clearly marking it as incomplete

null_email_count = users['email'].isnull().sum()
users['email'] = users['email'].fillna('unknown')

print(f'Null emails filled with "unknown": {null_email_count:,}')

Null emails filled with "unknown": 2,975


In [10]:
# ── Step 2.5: Fix null genders ───────────────────────────
null_gender_count = users['gender'].isnull().sum()
users['gender'] = users['gender'].fillna('Unknown')

print(f'Null genders filled with "Unknown": {null_gender_count:,}')

Null genders filled with "Unknown": 6,891


In [11]:
# ── Step 2.6: Standardise name formatting ────────────────
# .str.title() converts 'JOHN SMITH' or 'john smith' → 'John Smith'
# .str.strip() removes any leading/trailing spaces

users['full_name'] = users['full_name'].str.strip().str.title()

print('Sample names after cleaning:')
print(users['full_name'].sample(5).values)

Sample names after cleaning:
['Gabriela Sanchez' 'Dae Lim' 'Ngozi Okafor' 'Luke Jackson' 'Emily Allen']


In [12]:
# ── Step 2.7: Fix date column dtype ──────────────────────
# Right now signup_date is stored as a string (object)
# We convert it to a proper datetime type so we can do date maths later

users['signup_date'] = pd.to_datetime(users['signup_date'])

print(f'signup_date dtype after fix: {users["signup_date"].dtype}')

signup_date dtype after fix: datetime64[ns]


In [13]:
# ── Step 2.8: Final users audit ──────────────────────────
print('USERS TABLE — After Cleaning')
print(f'  Rows    : {len(users):,}')
print(f'  Nulls   : {users.isnull().sum().sum()}')
print(f'  Dupes   : {users.duplicated().sum()}')
print(f'\n  Dtypes:')
print(users.dtypes)

users.to_csv(f'{CLEAN_DIR}/users_clean.csv', index=False)
print(f'\n✅ Saved → {CLEAN_DIR}/users_clean.csv')

USERS TABLE — After Cleaning
  Rows    : 99,500
  Nulls   : 0
  Dupes   : 0

  Dtypes:
user_id                         int64
full_name                      object
email                          object
age                             int64
gender                         object
country                        object
currency                       object
signup_date            datetime64[ns]
acquisition_channel            object
user_segment                   object
dtype: object

✅ Saved → data/cleaned/users_clean.csv


---
## 3️⃣ Cleaning: `subscriptions` Table

**Problems:**
- Null plan_type and status (~2% each)
- end_date might be before start_date (data entry error)
- Suspicious monthly_price values (billing bugs)
- Date columns stored as strings

**Decisions:**
| Problem | Decision |
|---|---|
| Null plan_type | Fill with 'Free' (safest assumption) |
| Null status | Fill with 'Unknown' |
| end_date < start_date | Set end_date to NaN (we can't know the real date) |
| Price anomalies | Cap at max valid plan price (14.99 USD equivalent) |

In [14]:
# ── Step 3.1: Fix null plan_type and status ───────────────
subs['plan_type'] = subs['plan_type'].fillna('Free')
subs['status']    = subs['status'].fillna('Unknown')

print('plan_type distribution after cleaning:')
print(subs['plan_type'].value_counts())

plan_type distribution after cleaning:
plan_type
Free       39283
Basic      21442
Premium    19631
Student     7946
Family      7885
Duo         3813
Name: count, dtype: int64


In [15]:
# ── Step 3.2: Convert dates to datetime ──────────────────
# errors='coerce' means: if a date string is invalid, convert it to NaT
# NaT = 'Not a Time' — pandas equivalent of NaN for dates

subs['start_date'] = pd.to_datetime(subs['start_date'], errors='coerce')
subs['end_date']   = pd.to_datetime(subs['end_date'],   errors='coerce')

print(f'start_date dtype: {subs["start_date"].dtype}')
print(f'end_date dtype  : {subs["end_date"].dtype}')

start_date dtype: datetime64[ns]
end_date dtype  : datetime64[ns]


In [16]:
# ── Step 3.3: Fix end_date before start_date ─────────────
# This is a logical impossibility — subscription can't end before it started

# Only check rows where end_date is NOT null
bad_dates_mask = (
    subs['end_date'].notna() &
    (subs['end_date'] < subs['start_date'])
)

print(f'Rows with end_date before start_date: {bad_dates_mask.sum():,}')

# Set those end_dates to NaT (unknown)
subs.loc[bad_dates_mask, 'end_date'] = pd.NaT

print('Fixed — those end_dates set to NaT')

Rows with end_date before start_date: 0
Fixed — those end_dates set to NaT


In [17]:
# ── Step 3.4: Identify suspicious prices ─────────────────
# Valid max price is Family plan ~ 14.99 USD
# But we have multi-currency, so let's flag extreme outliers only
# We'll cap anything above 1000 (no plan costs more than ~$1000)

print('Price distribution before cleaning:')
print(subs['monthly_price'].describe())

suspicious = subs[subs['monthly_price'] > 1000]
print(f'\nRows with price > 1000: {len(suspicious):,}')

# Cap at 1000 — these are clearly data errors
subs.loc[subs['monthly_price'] > 1000, 'monthly_price'] = np.nan

print('Extreme prices replaced with NaN')

Price distribution before cleaning:
count    100000.000000
mean       5125.863780
std       26930.460379
min           0.000000
25%           0.000000
50%           4.990000
75%          24.950000
max      232345.000000
Name: monthly_price, dtype: float64

Rows with price > 1000: 11,345
Extreme prices replaced with NaN


In [18]:
# ── Step 3.5: Final subscriptions audit ──────────────────
print('SUBSCRIPTIONS TABLE — After Cleaning')
print(f'  Rows  : {len(subs):,}')
print(f'  Nulls : {subs.isnull().sum().sum():,}')
print(f'  Dupes : {subs.duplicated().sum()}')

subs.to_csv(f'{CLEAN_DIR}/subscriptions_clean.csv', index=False)
print(f'\n✅ Saved → {CLEAN_DIR}/subscriptions_clean.csv')

SUBSCRIPTIONS TABLE — After Cleaning
  Rows  : 100,000
  Nulls : 137,924
  Dupes : 0

✅ Saved → data/cleaned/subscriptions_clean.csv


---
## 4️⃣ Cleaning: `listening_events` Table

This is our **biggest table (1.1M+ rows)**. Problems to fix:
- Null genre (~2%) and null device (~3%)
- Outlier durations: -1, -99, 9999 seconds
- Date/time stored as strings

**Important:** For large tables, we must be careful about performance.
We use vectorised pandas operations (not loops) to keep things fast.

In [19]:
# ── Step 4.1: Check the scale of problems ─────────────────
print(f'Total rows      : {len(events):,}')
print(f'Null genre      : {events["genre"].isnull().sum():,}')
print(f'Null device     : {events["device"].isnull().sum():,}')

print('\nlisten_duration_sec — suspicious values:')
# Convert to numeric first (it might be object type due to our outlier injection)
events['listen_duration_sec'] = pd.to_numeric(events['listen_duration_sec'], errors='coerce')
print(events[events['listen_duration_sec'] < 0]['listen_duration_sec'].value_counts())
print(events[events['listen_duration_sec'] > 600]['listen_duration_sec'].value_counts().head())

Total rows      : 1,186,834
Null genre      : 23,573
Null device     : 35,508

listen_duration_sec — suspicious values:
listen_duration_sec
-1     114
-99     89
Name: count, dtype: int64
listen_duration_sec
9999    97
Name: count, dtype: int64


In [20]:
# ── Step 4.2: Fix null genre ──────────────────────────────
# We fill with 'Unknown' — we cannot guess what genre was played

events['genre'] = events['genre'].fillna('Unknown')
print(f'Genres after cleaning: {events["genre"].nunique()} unique values')

Genres after cleaning: 24 unique values


In [21]:
# ── Step 4.3: Fix null device ─────────────────────────────
events['device'] = events['device'].fillna('Unknown')
print(f'Devices after cleaning: {events["device"].value_counts().to_dict()}')

Devices after cleaning: {'Mobile': 621796, 'Desktop': 208900, 'Tablet': 91959, 'Smart TV': 87997, 'Web Browser': 57308, 'Smart Speaker': 57193, 'Unknown': 35508, 'Car Audio': 26173}


In [22]:
# ── Step 4.4: Fix outlier durations ──────────────────────
# Valid range: 0 seconds (skip) to 600 seconds (10 minutes — longest normal song)
# Negative values and 9999 are clearly errors

before_invalid = (events['listen_duration_sec'] < 0) | (events['listen_duration_sec'] > 600)
print(f'Invalid duration rows: {before_invalid.sum():,}')

# Replace invalid durations with NaN
events.loc[before_invalid, 'listen_duration_sec'] = np.nan

# Fill NaN durations with the median valid duration
median_dur = events['listen_duration_sec'].median()
events['listen_duration_sec'] = events['listen_duration_sec'].fillna(median_dur).astype(int)

print(f'Median duration used for fill: {median_dur:.0f} seconds')
print(f'Duration range after cleaning: {events["listen_duration_sec"].min()} - {events["listen_duration_sec"].max()} seconds')

Invalid duration rows: 300
Median duration used for fill: 162 seconds
Duration range after cleaning: 0 - 300 seconds


In [23]:
# ── Step 4.5: Convert event_date to datetime ─────────────
events['event_date'] = pd.to_datetime(events['event_date'], errors='coerce')

# Drop any rows where event_date became NaT (very rare bad dates)
before = len(events)
events = events.dropna(subset=['event_date'])
print(f'Rows dropped due to invalid dates: {before - len(events):,}')
print(f'Total events remaining: {len(events):,}')

Rows dropped due to invalid dates: 0
Total events remaining: 1,186,834


In [24]:
# ── Step 4.6: Final events audit ─────────────────────────
print('LISTENING_EVENTS TABLE — After Cleaning')
print(f'  Rows  : {len(events):,}')
print(f'  Nulls : {events.isnull().sum().sum():,}')
print(f'  Dupes : {events.duplicated().sum():,}')

# For large files, save with compression to reduce file size
events.to_csv(f'{CLEAN_DIR}/listening_events_clean.csv', index=False)
print(f'\n✅ Saved → {CLEAN_DIR}/listening_events_clean.csv')

LISTENING_EVENTS TABLE — After Cleaning
  Rows  : 1,186,834
  Nulls : 0
  Dupes : 0

✅ Saved → data/cleaned/listening_events_clean.csv


---
## 5️⃣ Cleaning: `payments` Table

**Problems:**
- Duplicate rows (~1.2%) — double-charge scenario
- Null payment_method (~3%)
- Suspicious amounts (billing errors)

**Note:** Duplicate payments are serious — in real companies this would trigger a refund process. 
We log them separately before removing.

In [25]:
# ── Step 5.1: Find and log duplicate payments ─────────────
# A real duplicate = same user_id, amount, AND payment_date
# (same person charged same amount on same day)

dup_mask   = pays.duplicated(subset=['user_id','amount','payment_date'], keep='first')
dup_pays   = pays[dup_mask].copy()

print(f'Duplicate payment rows found: {len(dup_pays):,}')
print('Sample duplicates:')
print(dup_pays[['user_id','amount','payment_date','status']].head())

# Save duplicates — in a real job, this would go to the finance team
dup_pays.to_csv(f'{CLEAN_DIR}/duplicate_payments_flagged.csv', index=False)

# Remove duplicates from main table
pays = pays[~dup_mask].copy()
print(f'\nPayments table after removing dupes: {len(pays):,} rows')

Duplicate payment rows found: 960
Sample duplicates:
       user_id  amount payment_date    status
80000    38494    9.85   2021-03-06   Success
80001    45042    4.57   2023-10-20   Success
80002    64358    3.48   2021-10-24   Success
80003    78382  413.68   2019-01-07   Success
80004    64991    4.55   2020-08-13  Refunded

Payments table after removing dupes: 80,000 rows


In [26]:
# ── Step 5.2: Fix null payment_method ────────────────────
pays['payment_method'] = pays['payment_method'].fillna('Unknown')
print('Payment method distribution:')
print(pays['payment_method'].value_counts())

Payment method distribution:
payment_method
Credit Card      17077
Debit Card       14073
UPI              13923
PayPal            9205
Apple Pay         6996
Google Pay        6247
Wallet            3780
Net Banking       3131
Unknown           2453
Bank Transfer     1606
Crypto            1509
Name: count, dtype: int64


In [27]:
# ── Step 5.3: Fix suspicious payment amounts ─────────────
print('Payment amount stats before cleaning:')
print(pays['amount'].describe())

# Flag and remove payments above 5000 (any currency) — clearly wrong
suspicious_pays = pays[pays['amount'] > 5000]
print(f'\nPayments > 5000: {len(suspicious_pays):,} rows')
pays = pays[pays['amount'] <= 5000].copy()

# Negative amounts are impossible
pays = pays[pays['amount'] >= 0].copy()

print(f'Payments remaining after cleaning: {len(pays):,}')

Payment amount stats before cleaning:
count     80000.000000
mean       8332.877533
std       33804.238796
min           0.490000
25%           5.390000
50%          14.210000
75%         448.820000
max      232345.500000
Name: amount, dtype: float64

Payments > 5000: 7,270 rows
Payments remaining after cleaning: 72,730


In [28]:
# ── Step 5.4: Convert payment_date to datetime ───────────
pays['payment_date'] = pd.to_datetime(pays['payment_date'], errors='coerce')
pays = pays.dropna(subset=['payment_date'])

print(f'payment_date dtype: {pays["payment_date"].dtype}')

pays.to_csv(f'{CLEAN_DIR}/payments_clean.csv', index=False)
print(f'\n✅ Saved → {CLEAN_DIR}/payments_clean.csv')

payment_date dtype: datetime64[ns]

✅ Saved → data/cleaned/payments_clean.csv


---
## 6️⃣ Cleaning: `support_tickets` Table

**Problems:**
- Null issue_type (~2%)
- resolved_date might be before created_date
- satisfaction_score filled for unresolved tickets (impossible)
- resolution_days might be negative

**This table is interesting** — it has *logical* inconsistencies, not just missing values.
Logical inconsistencies require you to think about the business rules.

In [29]:
# ── Step 6.1: Fix null issue_type ────────────────────────
tickets['issue_type'] = tickets['issue_type'].fillna('Unknown')
print('Issue type distribution:')
print(tickets['issue_type'].value_counts())

Issue type distribution:
issue_type
Billing Issue           3948
Playback Error          2942
Account Access          2364
Subscription Change     2049
App Crash               1539
Audio Quality           1337
Download Problem        1191
Offline Mode Issue       972
Family Plan Issue        753
Student Verification     749
Password Reset           601
Unknown                  385
Refund Request           381
Ads Not Removing         376
Other                    217
Data Privacy             196
Name: count, dtype: int64


In [30]:
# ── Step 6.2: Convert date columns ───────────────────────
tickets['created_date']  = pd.to_datetime(tickets['created_date'],  errors='coerce')
tickets['resolved_date'] = pd.to_datetime(tickets['resolved_date'], errors='coerce')

In [31]:
# ── Step 6.3: Fix resolved_date before created_date ──────
# Business rule: A ticket cannot be resolved before it was created!

bad_resolve = (
    tickets['resolved_date'].notna() &
    (tickets['resolved_date'] < tickets['created_date'])
)
print(f'Tickets with resolved before created: {bad_resolve.sum():,}')

tickets.loc[bad_resolve, 'resolved_date']  = pd.NaT
tickets.loc[bad_resolve, 'resolution_days'] = np.nan

print('Fixed — invalid resolved_dates set to NaT')

Tickets with resolved before created: 0
Fixed — invalid resolved_dates set to NaT


In [32]:
# ── Step 6.4: Fix satisfaction_score on unresolved tickets
# Business rule: You can't rate a ticket that was never resolved!

unresolved_with_score = (
    tickets['resolved_date'].isna() &
    tickets['satisfaction_score'].notna()
)
print(f'Unresolved tickets with a score: {unresolved_with_score.sum():,}')

tickets.loc[unresolved_with_score, 'satisfaction_score'] = np.nan
print('Fixed — scores removed from unresolved tickets')

Unresolved tickets with a score: 0
Fixed — scores removed from unresolved tickets


In [33]:
# ── Step 6.5: Fix negative resolution_days ───────────────
neg_days = tickets['resolution_days'] < 0
print(f'Negative resolution_days: {neg_days.sum():,}')
tickets.loc[neg_days, 'resolution_days'] = np.nan

tickets.to_csv(f'{CLEAN_DIR}/support_tickets_clean.csv', index=False)
print(f'\n✅ Saved → {CLEAN_DIR}/support_tickets_clean.csv')

Negative resolution_days: 0

✅ Saved → data/cleaned/support_tickets_clean.csv


---
## 7️⃣ Cleaning Summary Report

Always end a cleaning notebook with a clear before/after comparison.
This shows stakeholders and interviewers exactly what was done.

In [34]:
# ── Final Before vs After Report ─────────────────────────

# Reload raw files for comparison
raw_shapes = {
    'users'           : pd.read_csv(f'{RAW_DIR}/users.csv').shape,
    'subscriptions'   : pd.read_csv(f'{RAW_DIR}/subscriptions.csv').shape,
    'listening_events': pd.read_csv(f'{RAW_DIR}/listening_events.csv').shape,
    'payments'        : pd.read_csv(f'{RAW_DIR}/payments.csv').shape,
    'support_tickets' : pd.read_csv(f'{RAW_DIR}/support_tickets.csv').shape,
}

clean_shapes = {
    'users'           : users.shape,
    'subscriptions'   : subs.shape,
    'listening_events': events.shape,
    'payments'        : pays.shape,
    'support_tickets' : tickets.shape,
}

print(f'{'='*65}')
print(f'  CLEANING SUMMARY REPORT')
print(f'{'='*65}')
print(f'  {"Table":<22} {"Raw Rows":>12} {"Clean Rows":>12} {"Removed":>10}')
print(f'  {"-"*60}')

total_raw = total_clean = 0
for name in raw_shapes:
    r = raw_shapes[name][0]
    c = clean_shapes[name][0]
    total_raw   += r
    total_clean += c
    print(f'  {name:<22} {r:>12,} {c:>12,} {r-c:>10,}')

print(f'  {"-"*60}')
print(f'  {"TOTAL":<22} {total_raw:>12,} {total_clean:>12,} {total_raw-total_clean:>10,}')
print(f'{'='*65}')
print(f'\n  Cleaned files saved to: data/cleaned/')
print(f'  Ready for Phase 3 → Exploratory Data Analysis 🎉')

  CLEANING SUMMARY REPORT
  Table                      Raw Rows   Clean Rows    Removed
  ------------------------------------------------------------
  users                       101,500       99,500      2,000
  subscriptions               100,000      100,000          0
  listening_events          1,186,834    1,186,834          0
  payments                     80,960       72,730      8,230
  support_tickets              20,000       20,000          0
  ------------------------------------------------------------
  TOTAL                     1,489,294    1,479,064     10,230

  Cleaned files saved to: data/cleaned/
  Ready for Phase 3 → Exploratory Data Analysis 🎉


---
## ✅ Phase 2 Complete!

**What you accomplished in this notebook:**

1. **Audited** all 5 tables before touching anything
2. **Removed** duplicate rows and flagged them for business review
3. **Handled nulls** thoughtfully — different strategy for each column
4. **Fixed logical inconsistencies** — not just missing values
5. **Standardised** formats — dates, name casing, numeric types
6. **Isolated** bot accounts and duplicate payments into separate files
7. **Documented** every decision with comments

> 🧠 **Interview Tip:** If asked 'how do you handle missing values?', 
> never say 'I just delete them' or 'I fill with the mean'.
> Always say: *'It depends on the column, business context, and how much data is missing.'*
> This notebook shows exactly that thinking.

**Next → Phase 3: Exploratory Data Analysis (EDA)** 📊